# Data Analysis & Model Diagnostics Notebook
**Cross-Market Agricultural Commodity Prediction Engine**
CCDS / Pro-Ag Data Science Competition — Track 1

**Purpose:** This notebook generates all diagnostic figures for your data analyst to identify model improvement opportunities. Run all cells, review the figures, and note areas for improvement.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

# === DATA PATH — update if needed ===
DATA_ROOT = Path('../pro_ag_comp/ProAg_data')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('pro_ag_comp/ProAg_data')
print('Data root exists:', DATA_ROOT.exists())
print('Data root:', DATA_ROOT.resolve())

## 1. Data Inventory & Quality Assessment
Catalog every Track 1 file. **Analyst: check for unexpected row counts, missing files, or load errors.**

In [ ]:
import os
rows = []
for dirpath, _, filenames in os.walk(DATA_ROOT):
    for fn in filenames:
        fp = Path(dirpath) / fn
        if fn.startswith('.') or 'Track 2' in str(fp): continue
        if fn.endswith('.csv'):
            try:
                full = pd.read_csv(fp, low_memory=False)
                date_cols = [c for c in full.columns if 'date' in c.lower()]
                dr = ''
                if date_cols:
                    dates = pd.to_datetime(full[date_cols[0]], errors='coerce').dropna()
                    if not dates.empty: dr = f'{dates.min().date()} to {dates.max().date()}'
                nan_pct = full.isna().mean().mean() * 100
                all_nan = [c for c in full.select_dtypes(include=[np.number]).columns if full[c].isna().all()]
                rows.append({'File': fn, 'Rows': len(full), 'Cols': len(full.columns),
                             'Size_MB': round(fp.stat().st_size/1e6, 2), 'Date Range': dr,
                             'NaN %': f'{nan_pct:.1f}%', 'All-NaN Cols': ', '.join(all_nan) or 'None'})
            except:
                rows.append({'File': fn, 'Rows': '?', 'Cols': '?', 'Size_MB': round(fp.stat().st_size/1e6,2),
                             'Date Range': 'ERROR', 'NaN %': '?', 'All-NaN Cols': '?'})

manifest = pd.DataFrame(rows).sort_values('File')
print(f'Discovered {len(manifest)} Track 1 CSV files, {manifest["Size_MB"].sum():.1f} MB total')
manifest

## 2. Cash Cattle Deep Dive
**Analyst: Look for structural breaks, seasonality, and distribution shape. Are there regime changes the model might struggle with?**

In [ ]:
cash = pd.read_csv(DATA_ROOT / 'Cash Cattle.csv', low_memory=False)
cash['report_date'] = pd.to_datetime(cash['report_date'], errors='coerce')
cash = cash.sort_values('report_date')
print(f'Cash Cattle: {cash.shape[0]:,} rows, {cash.shape[1]} columns')
print(f'Date range: {cash["report_date"].min().date()} to {cash["report_date"].max().date()}')
print(f'Selling bases: {cash["selling_basis_description"].unique().tolist()}')
print(f'Grade descriptions: {cash["grade_description"].unique().tolist() if "grade_description" in cash.columns else "N/A"}')
cash.describe()

In [ ]:
# Fig 2a: Daily weighted average price with regime annotations
daily_cash = cash.groupby('report_date')['weighted_avg_price'].mean()
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

daily_cash.plot(ax=axes[0], color='#2563eb', linewidth=0.8)
axes[0].set_title('Cash Cattle — Daily Weighted Average Price ($/cwt)', fontsize=14)
axes[0].set_ylabel('Price ($)')
axes[0].axhline(daily_cash.mean(), color='red', linestyle='--', alpha=0.5, label=f'Mean: ${daily_cash.mean():.2f}')
axes[0].legend()

# Fig 2b: Rolling volatility (30-day std)
rolling_std = daily_cash.rolling(30).std()
rolling_std.plot(ax=axes[1], color='#dc2626', linewidth=0.8)
axes[1].set_title('30-Day Rolling Volatility', fontsize=14)
axes[1].set_ylabel('Std Dev ($)')
plt.tight_layout()
plt.show()

# Fig 2c: Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
daily_cash.hist(bins=50, ax=axes[0], color='#2563eb', edgecolor='white')
axes[0].set_title('Price Distribution')
axes[0].axvline(daily_cash.mean(), color='red', linestyle='--')

# Box plot by year
cash['year'] = cash['report_date'].dt.year
cash.boxplot(column='weighted_avg_price', by='year', ax=axes[1])
axes[1].set_title('Price by Year')
plt.suptitle('')
plt.tight_layout()
plt.show()

print(f'\n=== ANALYST NOTES ===')
print(f'Price range: ${daily_cash.min():.2f} to ${daily_cash.max():.2f}')
print(f'Mean: ${daily_cash.mean():.2f}, Median: ${daily_cash.median():.2f}')
print(f'Skewness: {daily_cash.skew():.3f}')
print(f'Max volatility period: {rolling_std.idxmax()}')

## 3. Cutout (Choice vs Select) Analysis
**Analyst: The Choice-Select spread is a key quality premium indicator. Watch for spread compression/expansion patterns.**

In [ ]:
cutout = pd.read_csv(DATA_ROOT / 'Cutout (Select_Choice).csv', low_memory=False)
cutout['report_date'] = pd.to_datetime(cutout['report_date'], errors='coerce')
if 'Attribute' in cutout.columns:
    cutout = cutout.rename(columns={'Attribute': 'grade_description', 'Value': 'wtd_avg'})
cutout = cutout.sort_values('report_date')

pivot = cutout.pivot_table(index='report_date', columns='grade_description', values='wtd_avg', aggfunc='mean')
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
pivot.plot(ax=axes[0], linewidth=0.8)
axes[0].set_title('Cutout Values Over Time ($/cwt)', fontsize=14)
axes[0].set_ylabel('Price ($)')

# Choice-Select spread
choice_col = next((c for c in pivot.columns if 'choice' in str(c).lower()), None)
select_col = next((c for c in pivot.columns if 'select' in str(c).lower()), None)
if choice_col and select_col:
    spread = pivot[choice_col] - pivot[select_col]
    spread.plot(ax=axes[1], color='#8b5cf6', linewidth=0.8)
    axes[1].axhline(0, color='gray', linestyle='--')
    axes[1].set_title('Choice-Select Spread ($/cwt)', fontsize=14)
    print(f'Spread stats: mean=${spread.mean():.2f}, std=${spread.std():.2f}')
plt.tight_layout()
plt.show()

## 4. Nearby Futures Analysis
**Analyst: Note that Nearby Futures contains HOG/PORK carcass data (~$80/cwt), NOT live cattle futures (~$200/cwt). This is critical for cross-market interpretation.**

In [ ]:
futures = pd.read_csv(DATA_ROOT / 'Nearby Futures.csv', low_memory=False)
futures['report_date'] = pd.to_datetime(futures['report_date'], errors='coerce')
futures = futures.sort_values('report_date')

fig, ax = plt.subplots(figsize=(14, 5))
for name in futures['Name'].dropna().unique():
    sub = futures[futures['Name'] == name]
    ax.plot(sub['report_date'], sub['price_5day'], label=name, linewidth=0.8)
ax.set_title('Nearby Futures — 5-Day Price by Region (HOG/PORK Carcass)', fontsize=14)
ax.set_ylabel('Price ($/cwt)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Regions: {futures["Name"].unique().tolist()}')
print(f'Purchase type: {futures["purchase_type"].unique().tolist()}')
print(f'Price range: ${futures["price_5day"].min():.2f} to ${futures["price_5day"].max():.2f}')

## 5. Production & Harvest Data
**Analyst: Production volumes can be leading indicators. Check for seasonal patterns and trend breaks.**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (fn, title) in enumerate([
    ('Beef Production.csv', 'Beef Production (LBS)'),
    ('Pork Production.csv', 'Pork Production'),
    ('Weekly Harvest.csv', 'Weekly Harvest Volume'),
    ('Carcass Weights.csv', 'Carcass Weights')]):
    ax = axes[i//2, i%2]
    fp = DATA_ROOT / fn
    if not fp.exists(): continue
    df = pd.read_csv(str(fp), low_memory=False)
    date_cols = [c for c in df.columns if 'date' in c.lower()]
    num_cols = df.select_dtypes(include=[np.number]).columns
    if date_cols and len(num_cols) > 0:
        df[date_cols[0]] = pd.to_datetime(df[date_cols[0]], errors='coerce')
        df = df.sort_values(date_cols[0])
        ax.plot(df[date_cols[0]], df[num_cols[0]], linewidth=0.8, color='#16a34a')
    ax.set_title(title, fontsize=12)
plt.tight_layout()
plt.show()

## 6. Futures Contract Curves
**Analyst: Check for contango/backwardation patterns across contract months.**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, commodity in enumerate(['Live Cattle', 'Hog', 'Corn', 'Soybeans']):
    ax = axes[idx//2, idx%2]
    d = DATA_ROOT / f'{commodity} 2025 Futures'
    if not d.exists(): ax.set_title(f'{commodity} — No data'); continue
    for f in sorted(d.glob('*.xlsx')):
        try:
            raw = pd.read_excel(str(f))
            if any('Unnamed' in str(c) for c in raw.columns):
                raw.columns = raw.iloc[0].astype(str)
                raw = raw.iloc[1:].reset_index(drop=True)
            if 'Time' in raw.columns and 'Close' in raw.columns:
                raw['Date'] = pd.to_datetime(raw['Time'], errors='coerce')
                raw['Close'] = pd.to_numeric(raw['Close'], errors='coerce')
                raw = raw.dropna(subset=['Date','Close']).sort_values('Date')
                ax.plot(raw['Date'], raw['Close'], label=f.stem.split(' ')[0], linewidth=0.7)
        except: pass
    ax.set_title(f'{commodity} 2025 Futures', fontsize=12)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 7. Cross-Market Correlation Analysis
**Analyst: Identify which cross-market features genuinely predict cash cattle vs. which are spurious. High correlation ≠ causation.**

In [ ]:
def load_and_prep(filename):
    fp = DATA_ROOT / filename
    if not fp.exists(): return pd.DataFrame()
    df = pd.read_csv(fp, low_memory=False)
    date_cols = [c for c in df.columns if 'date' in c.lower()]
    if not date_cols: return pd.DataFrame()
    df['Date_Index'] = pd.to_datetime(df[date_cols[0]], errors='coerce').dt.date
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols: return pd.DataFrame()
    base = filename.replace('.csv', '')
    df = df[['Date_Index'] + num_cols].rename(columns={c: f'{c} ({base})' for c in num_cols})
    return df.groupby('Date_Index').mean().reset_index()

datasets = ['Cash Cattle.csv', 'Cutout (Select_Choice).csv', 'Nearby Futures.csv',
            'Beef Production.csv', 'Carcass Weights.csv', 'Weekly Harvest.csv']
merged = pd.DataFrame()
for fn in datasets:
    temp = load_and_prep(fn)
    if temp.empty: continue
    if merged.empty: merged = temp
    else: merged = pd.merge(merged, temp, on='Date_Index', how='outer')

merged = merged.sort_values('Date_Index').dropna(axis=1, how='all').ffill().dropna()
print(f'Merged: {merged.shape[0]:,} rows × {merged.shape[1]} columns')

num = merged.select_dtypes(include=[np.number])
top_cols = num.std().nlargest(15).index.tolist()
corr = num[top_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, linewidths=0.5, ax=ax,
            xticklabels=[c[:30] for c in corr.columns], yticklabels=[c[:30] for c in corr.columns])
ax.set_title('Cross-Market Correlation Heatmap (Top 15 Features)', fontsize=14)
plt.tight_layout()
plt.show()

## 8. FRED Macroeconomic Indicators
**Analyst: These external indicators may improve model performance by capturing regime changes invisible in price history alone.**

In [ ]:
FRED_API_KEY = '137417237dd06cd5b46636ee9af306fe'
try:
    from fredapi import Fred
    fred = Fred(api_key=FRED_API_KEY)
    series = {'Corn PPI': 'WPU0223', 'WTI Oil': 'DCOILWTICO', 'Fed Funds': 'FEDFUNDS',
              'CPI': 'CPIAUCSL', 'USD Index': 'DTWEXBGS'}
    fig, axes = plt.subplots(len(series), 1, figsize=(14, 3*len(series)), sharex=True)
    fred_dfs = {}
    for i, (name, sid) in enumerate(series.items()):
        data = fred.get_series(sid, observation_start='2019-01-01')
        if data is not None and not data.empty:
            axes[i].plot(data.index, data.values, linewidth=0.8, color='#2563eb')
            axes[i].set_title(f'{name} ({sid})', fontsize=11)
            axes[i].set_ylabel('Value')
            fred_dfs[name] = data
            print(f'✅ {name}: latest={data.dropna().iloc[-1]:.4f} ({data.dropna().index[-1].date()})')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('fredapi not installed. Run: pip install fredapi')

## 9. XGBoost Model — Full Diagnostics
**Analyst: This is the core model. Review each diagnostic carefully:**
- **CV fold variance** — high variance = model is sensitive to time period
- **Feature importance** — lag features dominating means limited predictive horizon
- **Residual patterns** — systematic errors reveal improvable structure
- **Error by regime** — where does the model fail?

In [ ]:
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

target = 'weighted_avg_price (Cash Cattle)'
if target not in merged.columns:
    price_cols = [c for c in merged.columns if 'price' in c.lower()]
    target = price_cols[0] if price_cols else None
    print(f'Using fallback target: {target}')

ml = merged.drop(columns=['Date_Index']).copy()
for lag in [1, 3, 7, 14, 21]:
    ml[f'lag_{lag}'] = ml[target].shift(lag)
for w in [7, 14, 30]:
    ml[f'roll_mean_{w}'] = ml[target].shift(1).rolling(w).mean()
    ml[f'roll_std_{w}'] = ml[target].shift(1).rolling(w).std()
ml['roc_1'] = ml[target].pct_change(1)
ml['roc_7'] = ml[target].pct_change(7)
ml['momentum_7'] = ml[target] - ml[target].shift(1).rolling(7).mean()
ml = ml.dropna()

X = ml.drop(columns=[target])
y = ml[target]
print(f'Feature matrix: {X.shape[0]} rows × {X.shape[1]} features')
print(f'Target: {target}')

In [ ]:
# Time-series cross-validation with detailed per-fold analysis
tscv = TimeSeriesSplit(n_splits=5)
fold_results = []

fig, axes = plt.subplots(5, 1, figsize=(14, 20))
for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    m = xgb.XGBRegressor(n_estimators=150, learning_rate=0.08, max_depth=5, random_state=42)
    m.fit(Xtr, ytr, verbose=False)
    preds = m.predict(Xte)
    mae = mean_absolute_error(yte, preds)
    r2 = r2_score(yte, preds)
    mape = mean_absolute_percentage_error(yte, preds) * 100
    fold_results.append({'Fold': fold+1, 'MAE': mae, 'R2': r2, 'MAPE': mape,
                         'Train Size': len(Xtr), 'Test Size': len(Xte),
                         'Test Mean': yte.mean(), 'Test Std': yte.std()})
    
    axes[fold].plot(yte.values, label='Actual', color='#2563eb')
    axes[fold].plot(preds, label='Predicted', color='#f59e0b', linestyle='--')
    axes[fold].set_title(f'Fold {fold+1}: MAE={mae:.2f}, R²={r2:.4f}, MAPE={mape:.1f}%', fontsize=11)
    axes[fold].legend()

plt.suptitle('Cross-Validation Folds — Actual vs Predicted', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

cv_df = pd.DataFrame(fold_results)
print('\n=== CV RESULTS ===')
print(cv_df.to_string(index=False))
print(f'\nMean MAE: {cv_df["MAE"].mean():.4f} ± {cv_df["MAE"].std():.4f}')
print(f'Mean R²:  {cv_df["R2"].mean():.4f} ± {cv_df["R2"].std():.4f}')
print(f'Mean MAPE: {cv_df["MAPE"].mean():.2f}% ± {cv_df["MAPE"].std():.2f}%')
print(f'\n⚠️ ANALYST: High CV variance = model struggles with regime changes')

In [ ]:
# Final 80/20 split model
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

final_model = xgb.XGBRegressor(n_estimators=150, learning_rate=0.08, max_depth=5, random_state=42)
final_model.fit(X_train, y_train, verbose=False)
preds = final_model.predict(X_test)

print(f'Test MAE:  {mean_absolute_error(y_test, preds):.4f}')
print(f'Test MAPE: {mean_absolute_percentage_error(y_test, preds)*100:.2f}%')
print(f'Test R²:   {r2_score(y_test, preds):.4f}')

# Feature importance
importance = pd.DataFrame({'Feature': X.columns, 'Importance': final_model.feature_importances_})
importance = importance.sort_values('Importance', ascending=True).tail(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].barh(importance['Feature'].apply(lambda x: x[:40]), importance['Importance'], color='#2563eb')
axes[0].set_title('Top 20 Feature Importances', fontsize=14)
axes[0].set_xlabel('Importance (gain)')

# Actual vs Predicted
axes[1].plot(y_test.values, label='Actual', color='#2563eb')
axes[1].plot(preds, label='Predicted', color='#f59e0b', linestyle='--')
axes[1].set_title('Test Set: Actual vs Predicted', fontsize=14)
axes[1].legend()
plt.tight_layout()
plt.show()

## 10. Residual Analysis — Where Does the Model Fail?
**Analyst: This is the most important section. Look for:**
- Systematic bias (mean ≠ 0)
- Heteroscedasticity (errors growing with price level)
- Autocorrelated residuals (errors clustering in time)
- Regime-specific failures

In [ ]:
residuals = y_test.values - preds
dates_test = merged['Date_Index'].iloc[-len(y_test):].values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0,0].hist(residuals, bins=40, color='#8b5cf6', edgecolor='white')
axes[0,0].axvline(0, color='red', linestyle='--')
axes[0,0].set_title('Residual Distribution')
axes[0,0].set_xlabel('Residual (Actual - Predicted)')

# Residuals over time
axes[0,1].scatter(range(len(residuals)), residuals, alpha=0.5, s=10, color='#dc2626')
axes[0,1].axhline(0, color='gray', linestyle='--')
axes[0,1].set_title('Residuals Over Time (Test Set)')
axes[0,1].set_xlabel('Observation')

# Residuals vs predicted (heteroscedasticity check)
axes[1,0].scatter(preds, residuals, alpha=0.5, s=10, color='#2563eb')
axes[1,0].axhline(0, color='red', linestyle='--')
axes[1,0].set_title('Residuals vs Predicted Value')
axes[1,0].set_xlabel('Predicted')
axes[1,0].set_ylabel('Residual')

# Autocorrelation of residuals
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(pd.Series(residuals), ax=axes[1,1])
axes[1,1].set_title('Residual Autocorrelation')
plt.tight_layout()
plt.show()

print(f'Mean residual: {residuals.mean():.4f} (should be ~0)')
print(f'Std residual:  {residuals.std():.4f}')
print(f'Skewness: {pd.Series(residuals).skew():.4f}')
print(f'Max overpredict: ${abs(residuals.min()):.2f}')
print(f'Max underpredict: ${residuals.max():.2f}')
bias = 'UNDERESTIMATES' if residuals.mean() > 1 else 'OVERESTIMATES' if residuals.mean() < -1 else 'approximately unbiased'
print(f'\n⚠️ Model {bias} (mean residual = {residuals.mean():.2f})')

## 11. Improvement Opportunities Checklist

### Model Architecture
- [ ] Try different `max_depth` values (3, 5, 7, 10) — current is 5
- [ ] Try different `n_estimators` (100, 200, 500) — current is 150
- [ ] Try `learning_rate` grid (0.01, 0.05, 0.08, 0.15) — current is 0.08
- [ ] Add regularization (`reg_alpha`, `reg_lambda`) to reduce overfitting
- [ ] Try LightGBM or CatBoost as alternatives

### Feature Engineering
- [ ] Add day-of-week, month, quarter as categorical features
- [ ] Add year-over-year change features
- [ ] Try longer lag windows (21, 30, 60 days)
- [ ] Add Bollinger Bands (price ± 2σ of rolling mean)
- [ ] Add MACD (moving average convergence/divergence)
- [ ] Incorporate FRED macro indicators as features (see Section 8)

### Data Quality
- [ ] Investigate the Cutout `trend` column (100% NaN — is data available elsewhere?)
- [ ] Verify Nearby Futures is the right commodity for your use case
- [ ] Check for data entry errors in extreme price values
- [ ] Consider using selling-basis-specific models (LIVE vs DRESSED)

### Model Evaluation
- [ ] Add walk-forward validation (retrain monthly, test next month)
- [ ] Add prediction intervals (not just point forecasts)
- [ ] Compute directional accuracy (% of correct up/down predictions)
- [ ] Test model on held-out recent data not used in any training

### FRED Integration Opportunities
- [ ] Test which FRED series improve out-of-sample R²
- [ ] Try FRED data as interaction features with price lags
- [ ] Use CPI-adjusted (real) prices instead of nominal